# 3. Fairness

**Question:** does the model work equally well for women and men, younger and older patients, and both hospitals?

We use the cross-validated predictions from notebook 2 (each patient predicted by a model that never saw them).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

data = pd.read_csv("../data/predictions.csv")
print("Patients:", len(data))

## What we measure for each group
- **Patients:** group size. Small groups give less reliable numbers.
- **Share sick:** how common the disease is in the group.
- **Accuracy:** share of patients classified correctly.
- **Sensitivity:** of the **sick** patients, how many the model finds. A low value means sick patients are missed, the most dangerous error.
- **Specificity:** of the **healthy** patients, how many the model correctly calls healthy.

In [ ]:
def group_report(group_column):
    rows = []
    for group, part in data.groupby(group_column):
        sick = part[part["target"] == 1]
        healthy = part[part["target"] == 0]
        rows.append({
            "group": group,
            "patients": len(part),
            "share_sick": part["target"].mean(),
            "accuracy": (part["predicted"] == part["target"]).mean(),
            "sensitivity": (sick["predicted"] == 1).mean(),
            "specificity": (healthy["predicted"] == 0).mean(),
        })
    return pd.DataFrame(rows).round(3)

In [ ]:
print("By sex")
print(group_report("group_sex"))
print()
print("By age")
print(group_report("group_age"))
print()
print("By hospital")
print(group_report("hospital"))

## Sensitivity by group
Missing a sick patient is the most serious error, so we compare sensitivity across groups.

In [ ]:
reports = [group_report("group_sex"), group_report("group_age"), group_report("hospital")]
table = pd.concat(reports)

plt.figure(figsize=(7, 4))
plt.bar(table["group"], table["sensitivity"])
plt.ylabel("Sensitivity (sick patients found)")
plt.ylim(0, 1)
plt.xticks(rotation=30)
plt.title("How many sick patients the model finds, by group")
plt.tight_layout()
plt.show()

## What to look for
- A clearly **lower sensitivity** in one group means the model misses more sick patients in that group.
- Before concluding, check the **group size**: with few patients, a difference can be due to chance.